# CSC3043S Assignment 2 in Google Colab

Use **Runtime → Change runtime type → Hardware accelerator → GPU** before running the cells. Keep the browser tab open while inference runs. The Drive-backed checkpoint lets you resume if Colab disconnects.

In [ ]:
import os, sys, torch
print('Python:', sys.version)
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')
!nvidia-smi

In [ ]:
# Mount Google Drive. Put your assignment folder at:
# MyDrive/CSC3043S/assignment 2
from google.colab import drive
from pathlib import Path
import os

drive.mount('/content/drive')
PROJECT = Path('/content/drive/MyDrive/CSC3043S/assignment 2')
if not PROJECT.exists():
    PROJECT = Path('/content/drive/MyDrive/assignment 2')
assert PROJECT.exists(), 'Upload the assignment 2 folder to Google Drive first'
os.chdir(PROJECT)
print('Working directory:', Path.cwd())
print('Manifest exists:', Path('data/manifest.csv').exists())
print('Images exist:', Path('data/val2017').exists())
print('Annotations exist:', Path('data/instances_val2017.json').exists())

In [ ]:
# Install the assignment dependencies.
# Colab already includes torch; requirements are safe to install/upgrade here.
%pip install -q -r requirements.txt

import numpy, pandas, sklearn, transformers, PIL, pycocotools
print('numpy', numpy.__version__)
print('pandas', pandas.__version__)
print('sklearn', sklearn.__version__)
print('transformers', transformers.__version__)

In [ ]:
# If data/manifest.csv already exists, this cell is skipped.
from pathlib import Path
import pandas as pd

if not Path('data/manifest.csv').exists():
    !python scripts/build_dataset.py --annotations data/instances_val2017.json --images data/val2017 --output data/manifest.csv --n-images 200
else:
    print('Using existing manifest:', Path('data/manifest.csv').stat().st_size, 'bytes')

manifest = pd.read_csv('data/manifest.csv')
print(len(manifest), 'questions', manifest.image_id.nunique(), 'images')
print(manifest.question_type.value_counts())

In [ ]:
# Run Checkpoint B on GPU. This cell is resumable: rerun it after a Colab disconnect.
!python scripts/run_inference.py \
  --manifest data/manifest.csv \
  --images data/val2017 \
  --output data/inference_results.pkl \
  --pooling mean \
  --device cuda

In [ ]:
# Generate Q3/Q4, probes, plots, and the report draft.
!python scripts/summarize_results.py --results data/inference_results.pkl --output data/inference_summary.json
!python scripts/run_probes.py --results data/inference_results.pkl --split data/train_val_split.json --output data/layer_auroc.csv --strategy mean
!python scripts/make_plots.py --input data/layer_auroc.csv --output data/layer_auroc.png
!python scripts/generate_report.py --output report_draft.md

from pathlib import Path
print(Path('data/inference_summary.json').read_text())

In [ ]:
# Package only the generated artifacts. Because the project is in Drive, the files already persist.
from pathlib import Path
import zipfile

files = [
    'data/inference_results.pkl', 'data/inference_summary.json',
    'data/train_val_split.json', 'data/layer_auroc.csv', 'data/layer_auroc.png',
    'data/manifest.csv', 'report_draft.md'
]
existing = [Path(file) for file in files if Path(file).exists()]
print('Artifacts:', [str(file) for file in existing])
with zipfile.ZipFile('assignment2_results.zip', 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for file in existing:
        archive.write(file, arcname=str(file))
print('Created assignment2_results.zip')

from google.colab import files
files.download('assignment2_results.zip')